# Генерация признаков для LTV-предсказания (v3)

Исправления vs v2:
- **stride=30 дней** — нет перекрытия таргетов между фолдами
- **N_FOLDS=4** — честные фолды без утечки
- Сохраняются в `features_v3/`

In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
from datetime import date, timedelta
from typing import Optional

DATA_DIR = Path("../data/raw")
FEATURES_DIR = Path("../data/processed/features_v3")
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 4
STRIDE_DAYS = 30
BATCH_SIZE = 50_000
TARGET_COL = "gmv"

VALUE_COLS = [
    "gmv", "searches", "to_cart", "to_ord",
    "search_to_cart", "search_to_ord",
    "cat_to_cart", "cat_to_ord",
    "gmv_search", "gmv_cat",
    "search", "cat",
]

BINARY_COLS = [
    "has_search_to_cart", "has_search_to_ord",
    "has_cat_to_cart", "has_cat_to_ord",
]

WINDOWS = [
    ("3d",   2,   0),
    ("7d",   6,   0),
    ("14d",  13,  0),
    ("30d",  29,  0),
    ("60d",  59,  0),
    ("90d",  89,  0),
    ("180d", 179, 0),
]

In [ ]:
def generate_cv_anchor_dates(
    data: pl.DataFrame,
    prediction_horizon_days: int = 30,
    stride_days: int = STRIDE_DAYS,
    min_history_days: int = 180,
    n_folds: Optional[int] = None,
) -> list[date]:
    min_date = data["event_date"].min()
    max_date = data["event_date"].max()

    latest_anchor = max_date - timedelta(days=prediction_horizon_days)
    earliest_anchor = min_date + timedelta(days=min_history_days - 1)

    n_steps = (latest_anchor - earliest_anchor).days // stride_days
    all_anchors = [latest_anchor - timedelta(days=i * stride_days) for i in range(n_steps + 1)]
    all_anchors = sorted(all_anchors)

    if n_folds:
        return all_anchors[-n_folds:]
    return all_anchors

In [ ]:
def _build_feature_exprs(anchor_val: date, windows: list, value_cols: list, binary_cols: list) -> list[pl.Expr]:
    exprs = []
    
    for w_name, start_off, end_off in windows:
        w_start = anchor_val - timedelta(days=start_off)
        w_end = anchor_val - timedelta(days=end_off)
        mask = pl.col("event_date").is_between(w_start, w_end)

        for col in value_cols:
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(0.0).sum().alias(f"{col}_sum_{w_name}"))
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(None).max().alias(f"{col}_max_{w_name}"))
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(None).mean().alias(f"{col}_mean_{w_name}"))
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(None).std().alias(f"{col}_std_{w_name}"))
            
        exprs.append(pl.when(mask & (pl.col("searches") > 0)).then(1).otherwise(0).sum().alias(f"freq_search_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("to_ord") > 0)).then(1).otherwise(0).sum().alias(f"freq_ord_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("to_cart") > 0)).then(1).otherwise(0).sum().alias(f"freq_cart_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("cat") > 0)).then(1).otherwise(0).sum().alias(f"freq_cat_{w_name}"))
        exprs.append(pl.when(mask & ((pl.col("searches") > 0) | (pl.col("cat") > 0))).then(1).otherwise(0).sum().alias(f"active_days_{w_name}"))

        for bcol in binary_cols:
            exprs.append(pl.when(mask & (pl.col(bcol) > 0)).then(1).otherwise(0).sum().alias(f"freq_{bcol}_{w_name}"))

    last_ord_dt = pl.col("event_date").filter(pl.col("to_ord") > 0).max()
    last_search_dt = pl.col("event_date").filter(pl.col("searches") > 0).max()
    last_cart_dt = pl.col("event_date").filter(pl.col("to_cart") > 0).max()
    last_cat_dt = pl.col("event_date").filter(pl.col("cat") > 0).max()
    first_event_dt = pl.col("event_date").min()
    
    exprs.append((pl.lit(anchor_val) - last_ord_dt).dt.total_days().fill_null(-1).alias("recency_ord_days"))
    exprs.append((pl.lit(anchor_val) - last_search_dt).dt.total_days().fill_null(-1).alias("recency_search_days"))
    exprs.append((pl.lit(anchor_val) - last_cart_dt).dt.total_days().fill_null(-1).alias("recency_cart_days"))
    exprs.append((pl.lit(anchor_val) - last_cat_dt).dt.total_days().fill_null(-1).alias("recency_cat_days"))
    exprs.append((pl.lit(anchor_val) - first_event_dt).dt.total_days().fill_null(-1).alias("user_age_days"))
    
    return exprs


def _build_secondary_exprs() -> list[pl.Expr]:
    eps = 1.0
    exprs = []
    
    exprs.extend([
        (pl.col("gmv_sum_3d") / (pl.col("gmv_sum_7d") + eps)).alias("trend_gmv_3d_7d"),
        (pl.col("gmv_sum_7d") / (pl.col("gmv_sum_14d") + eps)).alias("trend_gmv_7d_14d"),
        (pl.col("gmv_sum_7d") / (pl.col("gmv_sum_30d") + eps)).alias("trend_gmv_7d_30d"),
        (pl.col("gmv_sum_14d") / (pl.col("gmv_sum_30d") + eps)).alias("trend_gmv_14d_30d"),
        (pl.col("gmv_sum_30d") / (pl.col("gmv_sum_90d") + eps)).alias("trend_gmv_30d_90d"),
        (pl.col("gmv_sum_30d") / (pl.col("gmv_sum_180d") + eps)).alias("trend_gmv_30d_180d"),
        (pl.col("gmv_sum_60d") / (pl.col("gmv_sum_180d") + eps)).alias("trend_gmv_60d_180d"),
    ])
    
    exprs.extend([
        (pl.col("searches_sum_3d") / (pl.col("searches_sum_7d") + eps)).alias("trend_search_3d_7d"),
        (pl.col("searches_sum_7d") / (pl.col("searches_sum_30d") + eps)).alias("trend_search_7d_30d"),
        (pl.col("searches_sum_30d") / (pl.col("searches_sum_90d") + eps)).alias("trend_search_30d_90d"),
    ])
    
    exprs.extend([
        (pl.col("to_ord_sum_7d") / (pl.col("to_ord_sum_30d") + eps)).alias("trend_ord_7d_30d"),
        (pl.col("to_ord_sum_30d") / (pl.col("to_ord_sum_90d") + eps)).alias("trend_ord_30d_90d"),
    ])

    exprs.extend([
        (pl.col("search_to_cart_sum_30d") / (pl.col("searches_sum_30d") + eps)).alias("cr_search_to_cart_30d"),
        (pl.col("search_to_ord_sum_30d") / (pl.col("searches_sum_30d") + eps)).alias("cr_search_to_ord_30d"),
        (pl.col("to_ord_sum_30d") / (pl.col("to_cart_sum_30d") + eps)).alias("cr_cart_to_ord_30d"),
        (pl.col("search_to_cart_sum_90d") / (pl.col("searches_sum_90d") + eps)).alias("cr_search_to_cart_90d"),
        (pl.col("to_ord_sum_90d") / (pl.col("to_cart_sum_90d") + eps)).alias("cr_cart_to_ord_90d"),
    ])
    
    exprs.extend([
        (pl.col("cat_to_cart_sum_30d") / (pl.col("cat_sum_30d") + eps)).alias("cr_cat_to_cart_30d"),
        (pl.col("cat_to_ord_sum_30d") / (pl.col("cat_sum_30d") + eps)).alias("cr_cat_to_ord_30d"),
        (pl.col("cat_to_cart_sum_90d") / (pl.col("cat_sum_90d") + eps)).alias("cr_cat_to_cart_90d"),
    ])
    
    exprs.extend([
        (pl.col("gmv_sum_7d") / (pl.col("to_ord_sum_7d") + eps)).alias("aov_7d"),
        (pl.col("gmv_sum_30d") / (pl.col("to_ord_sum_30d") + eps)).alias("aov_30d"),
        (pl.col("gmv_sum_90d") / (pl.col("to_ord_sum_90d") + eps)).alias("aov_90d"),
        (pl.col("gmv_sum_180d") / (pl.col("to_ord_sum_180d") + eps)).alias("aov_180d"),
    ])
    
    exprs.extend([
        (pl.col("gmv_cat_sum_30d") / (pl.col("gmv_search_sum_30d") + pl.col("gmv_cat_sum_30d") + eps)).alias("ratio_gmv_cat_30d"),
        (pl.col("gmv_cat_sum_90d") / (pl.col("gmv_search_sum_90d") + pl.col("gmv_cat_sum_90d") + eps)).alias("ratio_gmv_cat_90d"),
        (pl.col("cat_sum_30d") / (pl.col("search_sum_30d") + pl.col("cat_sum_30d") + eps)).alias("ratio_cat_actions_30d"),
    ])
    
    exprs.extend([
        (pl.col("searches_sum_30d") / (pl.col("active_days_30d") + eps)).alias("searches_per_active_day_30d"),
        (pl.col("gmv_sum_30d") / (pl.col("active_days_30d") + eps)).alias("gmv_per_active_day_30d"),
        (pl.col("to_cart_sum_30d") / (pl.col("active_days_30d") + eps)).alias("cart_per_active_day_30d"),
    ])
    
    for col in ["gmv", "searches", "to_ord"]:
        for w in ["30d", "90d"]:
            exprs.append(
                (pl.col(f"{col}_std_{w}") / (pl.col(f"{col}_mean_{w}").abs() + eps)).alias(f"{col}_cv_{w}")
            )
    
    return exprs

In [ ]:
def generate_features_and_targets(
    data: pl.DataFrame,
    anchor: date,
    user_batch: list[int],
    is_train: bool = True
) -> pl.DataFrame:
    
    max_back = max(w[1] for w in WINDOWS)
    
    data_f = data.filter(
        pl.col("user_id").is_in(user_batch) & 
        (pl.col("event_date") <= anchor) & 
        (pl.col("event_date") >= anchor - timedelta(days=max_back))
    )

    if len(data_f) > 0:
        features = (
            data_f.group_by("user_id")
            .agg(_build_feature_exprs(anchor, WINDOWS, VALUE_COLS, BINARY_COLS))
            .with_columns(_build_secondary_exprs())
            .with_columns(anchor_date=pl.lit(anchor))
        )
    else:
        features = pl.DataFrame({"user_id": user_batch, "anchor_date": anchor})
    
    index_df = pl.DataFrame({"user_id": user_batch}).with_columns(anchor_date=pl.lit(anchor))
    result = index_df.join(features, on=["anchor_date", "user_id"], how="left")
    
    feat_cols = [c for c in result.columns if c not in ["anchor_date", "user_id", 
                 "recency_ord_days", "recency_search_days", "recency_cart_days", 
                 "recency_cat_days", "user_age_days"]]
    result = result.with_columns([pl.col(c).fill_null(0.0) for c in feat_cols])

    if is_train:
        t_start = anchor + timedelta(days=1)
        t_end = anchor + timedelta(days=30)
        
        targets = (
            data.filter(
                pl.col("user_id").is_in(user_batch) & 
                pl.col("event_date").is_between(t_start, t_end)
            )
            .group_by("user_id")
            .agg(pl.col(TARGET_COL).sum().alias("target"))
        )
        result = result.join(targets, on="user_id", how="left").with_columns(pl.col("target").fill_null(0.0))
    else:
        result = result.with_columns(pl.lit(None).cast(pl.Float64).alias("target"))
        
    return result

In [ ]:
print("Чтение данных...")
data = pl.read_parquet(DATA_DIR / 'train.parquet')
user_ids = data["user_id"].unique().sort().to_list()
n_batches = (len(user_ids) + BATCH_SIZE - 1) // BATCH_SIZE

anchors_time_folds = generate_cv_anchor_dates(data, n_folds=N_FOLDS)
anchor_end_of_time = data["event_date"].max()

print(f"Трейн фолды ({N_FOLDS}): {anchors_time_folds}")
print(f"Stride: {STRIDE_DAYS} дней (без перекрытия таргетов)")
print(f"Тест anchor: {anchor_end_of_time}")
print(f"Пользователей: {len(user_ids):,} | Батчей: {n_batches}")

for fold_idx, anchor in enumerate(anchors_time_folds):
    fold_dir = FEATURES_DIR / f"fold_{fold_idx:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    
    for batch in range(n_batches):
        current_batch = user_ids[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE]
        out_df = generate_features_and_targets(data, anchor, current_batch, is_train=True)
        out_df.write_parquet(fold_dir / f"batch_{batch:04d}.parquet")
        
    print(f"Фолд {fold_idx} ({anchor}) готов.")

fold_dir = FEATURES_DIR / "fold_test"
fold_dir.mkdir(parents=True, exist_ok=True)

for batch in range(n_batches):
    current_batch = user_ids[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE]
    out_df = generate_features_and_targets(data, anchor_end_of_time, current_batch, is_train=False)
    out_df.write_parquet(fold_dir / f"batch_{batch:04d}.parquet")
    
print("Тестовый фолд готов!")